In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
import warnings

# 1. SILENZIAMO GLI AVVISI
warnings.filterwarnings('ignore')

# 2. CARICAMENTO DATI
df = pd.read_csv('listings.csv', on_bad_lines='skip', low_memory=False)

features_utili = [
    'guests', 'bedrooms', 'beds', 'baths', 
    'min_nights', 'num_reviews', 'rating_overall', 
    'ttm_avg_rate', 'room_type', 'city'
]
df_model = df[features_utili].copy()

# 3. FORZATURA NUMERICA E PULIZIA
numeric_features = ['guests', 'bedrooms', 'beds', 'baths', 'min_nights', 'num_reviews', 'rating_overall', 'ttm_avg_rate']

for col in numeric_features:
    df_model[col] = pd.to_numeric(df_model[col], errors='coerce')

df_model = df_model.dropna()
# NESSUN FILTRO SULLE ANOMALIE: Manteniamo il database originale integro.
print(f"✅ Dati puliti. Righe pronte per il modello: {len(df_model)}\n")

# 4. PRE-ELABORAZIONE
categorical_features = ['room_type', 'city']
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_features)
    ])

X_processed = preprocessor.fit_transform(df_model)

# 5. ADDESTRAMENTO K-MEANS
K_OTTIMALE = 4
kmeans_final = KMeans(n_clusters=K_OTTIMALE, random_state=42, n_init=10)
df_model['Cluster_ID'] = kmeans_final.fit_predict(X_processed)

# 6. MAPPATURA DEI NOMI (Con l'Anomalia inclusa)
mappa_nomi = {
    0: "Casa Vacanze (Famiglie)",
    1: "Villa di Lusso (Gruppi)",
    2: "Bestseller (Coppie/Singoli)",
    3: "Anomalia (Affitti Lungo Termine)" 
}

df_model['Categoria'] = df_model['Cluster_ID'].map(mappa_nomi)

# 7. STAMPA RISULTATI
print("🌟 PROFILO MEDIO PER CATEGORIA:")
cluster_summary = df_model.groupby('Categoria')[numeric_features].mean().round(2)
display(cluster_summary)

✅ Dati puliti. Righe pronte per il modello: 73810

🌟 PROFILO MEDIO PER CATEGORIA:


,guests,bedrooms,beds,baths,min_nights,num_reviews,rating_overall,ttm_avg_rate
Categoria,,,,,,,,
Anomalia (Affitti Lungo Termine),3.58,1.38,2.10,1.29,968.10,67.85,4.73,151.40
Bestseller (Coppie/Singoli),3.32,1.29,1.97,1.07,4.38,134.34,4.77,117.67
Casa Vacanze (Famiglie),6.13,2.71,4.28,1.75,4.34,73.47,4.76,224.65
Villa di Lusso (Gruppi),11.23,5.09,8.45,3.92,4.26,60.82,4.79,645.70
